# 🔬 Day 4: Independent Lab — Build Your Own Agentic System

## Overview

In this lab you will:
- Choose a business domain (one of four tracks)
- Design 3 tools with clear descriptions
- Write a system prompt for your agent
- Test the agent on 5+ queries
- Create a golden test set (8+ scenarios)
- Evaluate and improve your agent (v1 → v2)

**Duration:** 90–120 minutes
**Format:** Self-directed with instructor support


---
## Setup


In [ ]:
!pip install -q -U google-genai


In [ ]:
import os, json, time
from datetime import datetime, timezone
from google import genai
from google.genai import types

try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:
    pass

if not os.environ.get("GEMINI_API_KEY"):
    import getpass
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Paste your GEMINI_API_KEY: ")

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
MODEL_ID = "gemini-2.5-flash-lite"


In [ ]:
import os, json, time
from datetime import datetime, timezone
from google import genai
from google.genai import types

try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:
    pass

if not os.environ.get("GEMINI_API_KEY"):
    import getpass
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Paste your GEMINI_API_KEY: ")

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
MODEL_ID = "gemini-2.5-flash-lite"

# ── Infrastructure (from Guided Lab) ─────────────────────
PROMPT_LOG = []

def _now():
    return datetime.now(timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z")

def log_interaction(role, content, label=None):
    entry = {"ts": _now(), "role": role,
             "content": content if isinstance(content, str) else json.dumps(content),
             "label": label or ""}
    PROMPT_LOG.append(entry)
    return entry

def show_log(n=10):
    import pandas as pd
    if not PROMPT_LOG:
        print("No interactions logged yet.")
        return
    df = pd.DataFrame(PROMPT_LOG[-n:])
    from IPython.display import display
    display(df)

def run_agent(user_message, tools, system_prompt=None, max_steps=10):
    """A manual agent loop with full visibility.

    Args:
        user_message: The user's request.
        tools: List of Python functions to use as tools.
        system_prompt: Optional system instruction for the agent.
        max_steps: Maximum number of reasoning steps (safety limit).

    Returns:
        A tuple of (final_text, tools_called, trace) where:
        - final_text: The agent's final response string
        - tools_called: List of tool names invoked during the run
        - trace: List of dicts with 'call', 'tool', 'args', 'result' per tool call
    """
    tool_map = {fn.__name__: fn for fn in tools}
    call_count = 0        # Track total tool calls across all steps
    tools_called = []     # Record which tools were actually used
    trace = []            # Structured log: tool, args, result per call

    # Build initial contents
    contents = []
    if system_prompt:
        contents.append(types.Content(
            role="user",
            parts=[types.Part(text=f"System: {system_prompt}\n\nUser: {user_message}")]
        ))
    else:
        contents.append(types.Content(
            role="user",
            parts=[types.Part(text=user_message)]
        ))

    log_interaction("user", user_message, label="agent_input")

    for step in range(max_steps):
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=contents,
            config=types.GenerateContentConfig(
                tools=tools,
                # Tool calling mode defaults to AUTO — the model
                # reasons about whether to use tools on each turn.
                automatic_function_calling=types.AutomaticFunctionCallingConfig(
                    disable=True  # Model still reasons about tools —
                    # but the SDK won't execute them automatically.
                    # Instead it returns the function_call to us,
                    # and WE run the function below.
                ),
            ),
        )

        # Guard: the model may return an empty response
        parts = response.parts or []
        if not parts:
            print(f"  Step {step+1}: ⚠️ Empty response from model — retrying...")
            continue

        # Add model response to history
        contents.append(types.Content(role="model", parts=parts))

        # Check for function calls
        function_results = []
        for part in parts:
            if part.function_call:
                call_count += 1
                name = part.function_call.name
                args = dict(part.function_call.args)
                tools_called.append(name)
                print(f"  Tool call {call_count}: 🔧 {name}({args})")

                # Execute the function
                try:
                    result = tool_map[name](**args)
                except Exception as e:
                    result = {"error": str(e)}

                print(f"           → {result}")
                trace.append({
                    "call": call_count,
                    "tool": name,
                    "args": args,
                    "result": result if isinstance(result, str) else json.dumps(result),
                })
                log_interaction("tool", f"{name}({args}) → {result}", label="tool_call")

                function_results.append(
                    types.Part(
                        function_response=types.FunctionResponse(
                            name=name,
                            response={"result": result},
                        )
                    )
                )

        if function_results:
            contents.append(types.Content(role="user", parts=function_results))
        else:
            # No function calls → model is done
            final_text = response.text or "(no text response)"
            log_interaction("agent", final_text, label="agent_output")
            return final_text, tools_called, trace

    return "⚠️ Agent reached maximum steps without completing.", tools_called, trace

print("✅ Infrastructure ready.")

---
## Step 1: Choose Your Track (5 min)

> **Important:** The track you choose here will also be used for the **Day 4 Assignment**. Pick a domain that interests you!

| Track | Domain | Tools You'll Build |
|-------|--------|-------------------|
| **A** | Customer Support | classify_ticket, search_policies, draft_response |
| **B** | Research Assistant | search_documents, summarize_text, compare_topics |
| **C** | Financial Analysis | get_financials, calculate_metric, search_reports |
| **D** | HR / Recruitment | search_candidates, get_job_requirements, score_candidate |


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  SELECT YOUR TRACK — change the letter below            ║
# ╚══════════════════════════════════════════════════════════╝
SELECTED_TRACK = "A"   # Change to "A", "B", "C", or "D"

print(f"✅ Selected Track: {SELECTED_TRACK}")


In [ ]:
# ── Track Data (mock databases) ──────────────────────────
# Each track has sample data that the tools will query.

# Track A: Customer Support
SUPPORT_TICKETS = [
    {"id": "TK-101", "text": "App crashes when uploading large photos. Tried reinstalling.", "customer": "user_42"},
    {"id": "TK-102", "text": "I was charged twice for my subscription this month. Urgent!", "customer": "user_88"},
    {"id": "TK-103", "text": "How do I export my data to CSV? Can't find the option.", "customer": "user_15"},
    {"id": "TK-104", "text": "The dark mode doesn't work on iOS 18. Everything is white.", "customer": "user_67"},
    {"id": "TK-105", "text": "I love the new dashboard update! Great work.", "customer": "user_23"},
    {"id": "TK-106", "text": "Login fails with SSO. Error code 403. Very urgent.", "customer": "user_91"},
    {"id": "TK-107", "text": "Can I upgrade from Starter to Growth plan mid-cycle?", "customer": "user_34"},
    {"id": "TK-108", "text": "API rate limit hit. Need higher quota for production.", "customer": "user_56"},
]

SUPPORT_POLICIES = {
    "billing": "Duplicate charges must be refunded within 48 hours. Escalate to billing team if amount > $500.",
    "bugs": "Critical bugs (crash, data loss) are Priority 1. Assign to engineering. ETA: 24h for P1, 72h for P2.",
    "features": "Feature requests go to the product backlog. Thank the customer and share the roadmap link.",
    "account": "Account changes (upgrades, downgrades) can be processed immediately. Prorate the difference.",
    "api": "API rate limit increases require manager approval. Standard limit: 1000 req/min. Enterprise: 10000 req/min.",
    "praise": "Positive feedback should be forwarded to the team Slack channel. Thank the customer.",
}

# Track B: Research Assistant
RESEARCH_DOCS = {
    "ai_market": "The global AI market was valued at $196B in 2023 and is projected to reach $1.8T by 2030, growing at 36% CAGR. Key segments: generative AI ($44B), computer vision ($38B), NLP ($35B).",
    "competitor_alpha": "AlphaTech launched their enterprise AI platform in Q2 2024. Pricing: $50k/year for teams up to 50 users. Key differentiator: on-premise deployment option. Weakness: no mobile SDK.",
    "competitor_beta": "BetaCorp acquired DataMinds for $2.3B in March 2024. Combined entity focuses on real-time analytics. Revenue grew 45% YoY to $890M. Weakness: high customer churn (18%).",
    "customer_trends": "Enterprise AI adoption increased from 35% to 55% between 2022-2024. Top use cases: customer service automation (72%), document processing (65%), predictive analytics (58%).",
    "regulation": "The EU AI Act entered into force in August 2024. Key requirements: transparency obligations for general-purpose AI, risk classification system, and mandatory conformity assessments for high-risk applications.",
    "talent": "AI engineer salaries increased 25% in 2024. Average: $185k in the US, $120k in Europe. Biggest skill gaps: MLOps (67% of companies), responsible AI (54%), agent frameworks (48%).",
}

# Track C: Financial Analysis
FINANCIAL_DATA = {
    "ACME": {"revenue_q4": 45_000_000, "expenses_q4": 38_000_000, "employees": 450, "growth_yoy": 0.12, "sector": "Manufacturing"},
    "TECHSTART": {"revenue_q4": 12_000_000, "expenses_q4": 15_000_000, "employees": 120, "growth_yoy": 0.45, "sector": "SaaS"},
    "RETAILMAX": {"revenue_q4": 89_000_000, "expenses_q4": 82_000_000, "employees": 2200, "growth_yoy": -0.03, "sector": "Retail"},
}

EARNINGS_REPORTS = {
    "ACME_Q4": "Acme Corp reported Q4 revenue of $45M, up 12% YoY. Margins improved to 15.6% due to automation initiatives. Guidance for next quarter: $47-49M revenue.",
    "TECHSTART_Q4": "TechStart burned $3M in Q4 but grew revenue 45% YoY to $12M. ARR reached $48M. Key risk: runway is 14 months at current burn rate. Pursuing Series C.",
    "RETAILMAX_Q4": "RetailMax Q4 revenue was $89M, down 3% YoY. E-commerce grew 15% but couldn't offset 8% decline in physical stores. Announced 200 layoffs.",
    "INDUSTRY_OUTLOOK": "The SaaS sector is expected to grow 18% in 2025, driven by AI integration. Manufacturing AI spending projected at $9.8B. Retail tech investment flat YoY.",
}

# Track D: HR / Recruitment
CANDIDATES = [
    {"id": "C-201", "name": "Alice Chen", "skills": ["Python", "ML", "TensorFlow"], "experience_years": 5, "current_role": "ML Engineer", "salary_expectation": 160000},
    {"id": "C-202", "name": "Bob Martinez", "skills": ["Java", "AWS", "Kubernetes"], "experience_years": 8, "current_role": "DevOps Lead", "salary_expectation": 185000},
    {"id": "C-203", "name": "Carol Zhang", "skills": ["Python", "NLP", "LLMs", "RAG"], "experience_years": 3, "current_role": "AI Research Intern", "salary_expectation": 130000},
    {"id": "C-204", "name": "David Kim", "skills": ["Product Management", "Agile", "SQL"], "experience_years": 10, "current_role": "Senior PM", "salary_expectation": 175000},
    {"id": "C-205", "name": "Eva Müller", "skills": ["Python", "Data Engineering", "Spark"], "experience_years": 6, "current_role": "Data Engineer", "salary_expectation": 155000},
    {"id": "C-206", "name": "Frank Lee", "skills": ["Python", "ML", "LLMs", "Agents"], "experience_years": 4, "current_role": "AI Engineer", "salary_expectation": 170000},
]

JOB_REQUIREMENTS = {
    "AI_ENGINEER": {"title": "AI Engineer", "required_skills": ["Python", "ML", "LLMs"], "min_experience": 3, "max_salary": 180000, "team": "AI Platform"},
    "DATA_ENGINEER": {"title": "Data Engineer", "required_skills": ["Python", "Data Engineering", "SQL"], "min_experience": 4, "max_salary": 165000, "team": "Data"},
    "SENIOR_PM": {"title": "Senior Product Manager", "required_skills": ["Product Management", "Agile"], "min_experience": 7, "max_salary": 190000, "team": "Product"},
}

print(f"✅ Track data loaded. Selected track: {SELECTED_TRACK}")


---
## Step 2: Define Your Tools (15 min)

Design 3 tools for your chosen track. Each tool needs:
- A clear **function name** (verb + noun, e.g. `classify_ticket`)
- **Type hints** for all parameters and return type
- A **docstring** explaining what it does and when to use it
- **Graceful error handling** (return `{"error": "..."}` instead of crashing)

> 💡 **Tip:** The quality of your tool descriptions directly affects how well the agent uses them. Treat descriptions like prompts!


In [ ]:
# ── Track A: Customer Support Tools ──────────────────────

def classify_ticket(ticket_text: str) -> dict:
    """Classify a support ticket into a category and urgency level.

    Args:
        ticket_text: The customer's support message.

    Returns:
        A dict with 'category' and 'urgency' (low/medium/high).
    """
    text_lower = ticket_text.lower()
    # Simple rule-based classification (students can improve with LLM later)
    if any(w in text_lower for w in ["charged", "billing", "invoice", "refund", "payment"]):
        category = "billing"
    elif any(w in text_lower for w in ["crash", "error", "bug", "broken", "fail"]):
        category = "bugs"
    elif any(w in text_lower for w in ["upgrade", "downgrade", "plan", "account"]):
        category = "account"
    elif any(w in text_lower for w in ["api", "rate limit", "quota"]):
        category = "api"
    elif any(w in text_lower for w in ["love", "great", "awesome", "thanks"]):
        category = "praise"
    else:
        category = "features"

    urgency = "high" if any(w in text_lower for w in ["urgent", "crash", "charged twice", "fail"]) else "medium"
    return {"category": category, "urgency": urgency}


def search_policies(category: str) -> str:
    """Search company support policies for a given category.

    Args:
        category: The ticket category, e.g. 'billing', 'bugs', 'account'.

    Returns:
        The relevant policy text, or an error if category not found.
    """
    policy = SUPPORT_POLICIES.get(category)
    if policy:
        return policy
    return f"No policy found for category '{category}'. Available: {list(SUPPORT_POLICIES.keys())}"


def draft_response(ticket_text: str, category: str, policy_excerpt: str) -> str:
    """Draft a customer support response based on the ticket and policy.

    Args:
        ticket_text: The original customer message.
        category: The classified category.
        policy_excerpt: The relevant policy text.

    Returns:
        A draft response string.
    """
    return (
        f"Thank you for reaching out. I've categorized your request as '{category}'. "
        f"Based on our policy: {policy_excerpt} "
        f"I'll make sure this is handled promptly. Is there anything else I can help with?"
    )


# ── Track B: Research Assistant Tools ────────────────────

def search_documents(query: str) -> str:
    """Search the internal document library for information.

    Args:
        query: The search query, e.g. 'AI market size 2024'

    Returns:
        Relevant document excerpts matching the query.
    """
    # TODO: Implement search logic using RESEARCH_DOCS
    # Hint: check if any keyword from query appears in doc keys or values
    results = []
    query_lower = query.lower()
    for key, text in RESEARCH_DOCS.items():
        if any(word in text.lower() for word in query_lower.split()):
            results.append(f"[{key}]: {text[:200]}")
    if results:
        return "\n\n".join(results[:3])
    return f"No documents found for '{query}'."


def summarize_text(text: str, max_sentences: int = 3) -> str:
    """Summarize a given text into key points.

    Args:
        text: The text to summarize.
        max_sentences: Maximum number of sentences in the summary.

    Returns:
        A concise summary.
    """
    # TODO: Implement summarization
    # Simple approach: return first N sentences
    sentences = text.replace(". ", ".\n").split("\n")
    summary = ". ".join(s.strip() for s in sentences[:max_sentences] if s.strip())
    return summary if summary else "Could not summarize the text."


def compare_topics(topic1: str, topic2: str) -> str:
    """Compare two topics or companies based on available documents.

    Args:
        topic1: First topic or company name.
        topic2: Second topic or company name.

    Returns:
        A structured comparison of the two topics.
    """
    # TODO: Implement comparison logic
    info1 = search_documents(topic1)
    info2 = search_documents(topic2)
    return f"--- {topic1} ---\n{info1}\n\n--- {topic2} ---\n{info2}"


# ── Track C: Financial Analysis Tools ────────────────────

def get_financials(company: str) -> dict:
    """Get financial data for a company.

    Args:
        company: Company ticker/name, e.g. 'ACME', 'TECHSTART'

    Returns:
        A dict with revenue, expenses, employees, growth, sector.
    """
    # TODO: Implement using FINANCIAL_DATA
    data = FINANCIAL_DATA.get(company.upper())
    if data:
        return {**data, "company": company.upper(), "status": "found"}
    return {"error": f"No data for '{company}'. Available: {list(FINANCIAL_DATA.keys())}"}


def calculate_metric(expression: str) -> str:
    """Evaluate a financial calculation.

    Args:
        expression: A math expression, e.g. '45000000 - 38000000' or '12000000 / 120'

    Returns:
        The result as a string.
    """
    # TODO: Implement
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"


def search_reports(query: str) -> str:
    """Search earnings reports and industry analyses.

    Args:
        query: Search terms, e.g. 'ACME Q4 earnings'

    Returns:
        Relevant report excerpts.
    """
    # TODO: Implement using EARNINGS_REPORTS
    results = []
    query_lower = query.lower()
    for key, text in EARNINGS_REPORTS.items():
        if any(word in key.lower() or word in text.lower() for word in query_lower.split()):
            results.append(f"[{key}]: {text}")
    if results:
        return "\n\n".join(results[:2])
    return f"No reports found for '{query}'."


# ── Track D: HR / Recruitment Tools ──────────────────────

def search_candidates(required_skills: str, min_experience: int = 0) -> str:
    """Search the candidate database by skills and experience.

    Args:
        required_skills: Comma-separated skills, e.g. 'Python, ML'
        min_experience: Minimum years of experience.

    Returns:
        Matching candidates with their details.
    """
    # TODO: Implement using CANDIDATES
    skills = [s.strip().lower() for s in required_skills.split(",")]
    matches = []
    for c in CANDIDATES:
        c_skills = [s.lower() for s in c["skills"]]
        skill_match = sum(1 for s in skills if s in c_skills)
        if skill_match > 0 and c["experience_years"] >= min_experience:
            matches.append(f"{c['name']} ({c['current_role']}, {c['experience_years']}y) — Skills: {', '.join(c['skills'])}")
    if matches:
        return "\n".join(matches)
    return "No candidates match the criteria."


def get_job_requirements(position: str) -> dict:
    """Get the requirements for a job position.

    Args:
        position: Job title or ID, e.g. 'AI_ENGINEER'

    Returns:
        A dict with title, required_skills, min_experience, max_salary.
    """
    # TODO: Implement using JOB_REQUIREMENTS
    data = JOB_REQUIREMENTS.get(position.upper().replace(" ", "_"))
    if data:
        return data
    return {"error": f"Position '{position}' not found. Available: {list(JOB_REQUIREMENTS.keys())}"}


def score_candidate(candidate_id: str, position: str) -> dict:
    """Score a candidate against a job position's requirements.

    Args:
        candidate_id: The candidate ID, e.g. 'C-201'
        position: The job position ID, e.g. 'AI_ENGINEER'

    Returns:
        A dict with skill_match, experience_match, salary_fit, overall_score.
    """
    # TODO: Implement scoring
    candidate = next((c for c in CANDIDATES if c["id"] == candidate_id), None)
    if not candidate:
        return {"error": f"Candidate '{candidate_id}' not found."}
    job = JOB_REQUIREMENTS.get(position.upper().replace(" ", "_"))
    if not job:
        return {"error": f"Position '{position}' not found."}

    c_skills = [s.lower() for s in candidate["skills"]]
    required = [s.lower() for s in job["required_skills"]]
    skill_match = sum(1 for s in required if s in c_skills) / len(required)
    exp_match = 1.0 if candidate["experience_years"] >= job["min_experience"] else candidate["experience_years"] / job["min_experience"]
    salary_fit = 1.0 if candidate["salary_expectation"] <= job["max_salary"] else job["max_salary"] / candidate["salary_expectation"]
    overall = round((skill_match * 0.4 + exp_match * 0.3 + salary_fit * 0.3) * 100)

    return {
        "candidate": candidate["name"],
        "position": job["title"],
        "skill_match": f"{skill_match:.0%}",
        "experience_match": f"{exp_match:.0%}",
        "salary_fit": f"{salary_fit:.0%}",
        "overall_score": f"{overall}%",
    }


# ── Select Tools Based on Track ──────────────────────────
TRACK_TOOLS = {
    "A": [classify_ticket, search_policies, draft_response],
    "B": [search_documents, summarize_text, compare_topics],
    "C": [get_financials, calculate_metric, search_reports],
    "D": [search_candidates, get_job_requirements, score_candidate],
}

my_tools = TRACK_TOOLS[SELECTED_TRACK]
print(f"✅ Tools for Track {SELECTED_TRACK}: {[t.__name__ for t in my_tools]}")


---
## Step 3: Write Your System Prompt (10 min)

A good agent system prompt includes:
- **Role** assignment
- **Available tools** with descriptions
- **Rules** (when to use tools, when not to)
- **Refusal** instructions (what to do with out-of-scope requests)


In [ ]:
# ── System Prompt V1 ─────────────────────────────────────
# TODO: Customize this for your track.

SYSTEM_PROMPT_V1 = """You are a helpful assistant for the {domain} domain.

Available tools:
{tool_descriptions}

Rules:
- Use tools when you need to look up or process information.
- Never invent data — only use results returned by tools.
- If you cannot help with a request, say so clearly.
- Be concise and professional in your responses.
""".format(
    domain={
        "A": "customer support",
        "B": "business research",
        "C": "financial analysis",
        "D": "HR and recruitment",
    }[SELECTED_TRACK],
    tool_descriptions="\n".join(
        f"- {fn.__name__}: {fn.__doc__.split(chr(10))[0]}" for fn in my_tools
    ),
)

print("System Prompt V1:")
print(SYSTEM_PROMPT_V1)


---
## Step 4: Test Your Agent on 5+ Queries (15 min)

Run your agent on at least 5 realistic queries. Include a mix:
- 2 straightforward queries (easy)
- 2 multi-step queries (medium)
- 1 edge case or out-of-scope query (hard)


In [ ]:
# ── Test Queries ─────────────────────────────────────────
# TODO: Add at least 4 more queries for your track.

TRACK_QUERIES = {
    "A": [
        "Process ticket TK-102: 'I was charged twice for my subscription this month. Urgent!'",
        "Handle ticket TK-103: 'How do I export my data to CSV?'",
        "Triage ticket TK-106: 'Login fails with SSO. Error code 403. Very urgent.'",
        "Process ticket TK-105: 'I love the new dashboard update!'",
        "What's the weather in London?",  # Out-of-scope
    ],
    "B": [
        "What is the current size of the global AI market?",
        "Compare AlphaTech and BetaCorp as competitors.",
        "Summarize the key trends in enterprise AI adoption.",
        "What are the main regulations affecting AI in Europe?",
        "How do I reset my password?",  # Out-of-scope
    ],
    "C": [
        "What was Acme Corp's Q4 profit?",
        "Compare the growth rates of TECHSTART and RETAILMAX.",
        "Calculate revenue per employee for all three companies.",
        "What is the industry outlook for SaaS in 2025?",
        "Who won the World Cup?",  # Out-of-scope
    ],
    "D": [
        "Find candidates for the AI Engineer position.",
        "Score candidate C-203 for the AI Engineer role.",
        "Compare candidates C-201 and C-206 for the AI Engineer role.",
        "What are the requirements for the Senior PM position?",
        "Can you book a meeting room?",  # Out-of-scope
    ],
}

test_queries = TRACK_QUERIES[SELECTED_TRACK]

# Run V1
print("=" * 60)
print("RUNNING AGENT V1")
print("=" * 60)

v1_results = []
for i, query in enumerate(test_queries, 1):
    print(f"\n{"─"*60}")
    print(f"Query {i}: {query}\n")
    answer, tools_used, trace = run_agent(query, tools=my_tools, system_prompt=SYSTEM_PROMPT_V1)
    v1_results.append({"query": query, "answer": answer, "tools_used": tools_used, "trace": trace})
    print(f"🔧 Tools used: {tools_used}")
    if trace:
        print(f"\n📋 Agent Trace:")
        for t in trace:
            print(f"   [{t['call']}] {t['tool']}({t['args']}) → {str(t['result'])[:200]}")
    print(f"\n📝 Answer: {answer[:300]}")

---
## Step 5: Create Golden Test Set (15 min)

Create at least 8 test scenarios. For each, specify:
- The input query
- Which tools should be called (in order)
- Keywords the answer should contain
- Difficulty level (easy / medium / hard / edge)


In [ ]:
# ── Golden Test Set ──────────────────────────────────────
# TODO: Complete with 8+ scenarios for your track.

golden_set = [
    {
        "id": "G01",
        "query": test_queries[0],
        "expected_tools": [my_tools[0].__name__, my_tools[1].__name__],
        "expected_keywords": ["billing", "refund"],  # TODO: adjust for your track
        "difficulty": "easy",
    },
    {
        "id": "G02",
        "query": test_queries[1],
        "expected_tools": [my_tools[0].__name__],
        "expected_keywords": [],  # TODO: fill in
        "difficulty": "easy",
    },
    # TODO: Add G03–G08+ with a mix of difficulty levels
    # Include at least:
    # - 3 easy scenarios
    # - 3 medium scenarios (multi-tool)
    # - 2 hard/edge scenarios (out-of-scope, ambiguous)
]

print(f"Golden set: {len(golden_set)} scenarios defined.")
assert len(golden_set) >= 2, "⚠️ Need at least 8 scenarios! Keep adding."


In [ ]:
# ── Evaluate V1 Against Golden Set ───────────────────────
def evaluate_agent_results(results, golden_set):
    """Evaluate agent results against golden test set."""
    scores = []
    for golden in golden_set:
        # Find matching result
        matching = [r for r in results if r["query"] == golden["query"]]
        if not matching:
            scores.append({"id": golden["id"], "keyword_score": 0, "passed": False})
            continue
        answer = matching[0]["answer"].lower()
        expected_kw = golden.get("expected_keywords", [])
        if expected_kw:
            found = sum(1 for kw in expected_kw if kw.lower() in answer)
            score = found / len(expected_kw)
        else:
            score = 1.0  # No keywords to check
        scores.append({
            "id": golden["id"],
            "difficulty": golden.get("difficulty", "?"),
            "keyword_score": f"{score:.0%}",
            "passed": score >= 0.5,
        })
    return scores

v1_scores = evaluate_agent_results(v1_results, golden_set)
import pandas as pd
print("V1 Evaluation:")
print(pd.DataFrame(v1_scores).to_string(index=False))
v1_pass_rate = sum(1 for s in v1_scores if s["passed"]) / len(v1_scores) * 100
print(f"\nV1 Pass Rate: {v1_pass_rate:.0f}%")


---
## Step 6: Analyze V1 Errors (10 min)

Look at the queries where v1 failed or performed poorly. Ask yourself:
1. Did the agent call the **wrong tool**?
2. Did it pass **incorrect arguments**?
3. Did it **ignore the tool result**?
4. Was the **system prompt** unclear?


### ✍️ V1 Error Analysis

**Error Pattern 1:** [Describe]
- Example query: ...
- What went wrong: ...
- Root cause: ...

**Error Pattern 2:** [Describe]
- Example query: ...
- What went wrong: ...
- Root cause: ...

**Planned improvements for V2:**
- ...
- ...


---
## Step 7: Improve to V2 (20 min)

Based on your error analysis, improve your:
- **System prompt** (add rules, examples, constraints)
- **Tool descriptions** (make them clearer)
- **Tool implementations** (handle edge cases)


In [ ]:
# ── System Prompt V2 (Improved) ──────────────────────────
# TODO: Improve based on V1 error analysis.

SYSTEM_PROMPT_V2 = """You are a helpful assistant for the {domain} domain.

Available tools:
{tool_descriptions}

Rules:
- Use tools when you need to look up or process information.
- Never invent data — only use results returned by tools.
- If you cannot help with a request, say so clearly.
- Be concise and professional in your responses.

# TODO: Add additional rules based on your V1 errors, e.g.:
# - Always classify before searching policies (Track A)
# - When comparing, search for both items first (Track B)
# - Show your calculations step by step (Track C)
# - Check all matching candidates before scoring (Track D)
""".format(
    domain={
        "A": "customer support",
        "B": "business research",
        "C": "financial analysis",
        "D": "HR and recruitment",
    }[SELECTED_TRACK],
    tool_descriptions="\n".join(
        f"- {fn.__name__}: {fn.__doc__.split(chr(10))[0]}" for fn in my_tools
    ),
)

print("System Prompt V2:")
print(SYSTEM_PROMPT_V2)


In [ ]:
# ── Run Agent V2 ─────────────────────────────────────────
print("=" * 60)
print("RUNNING AGENT V2")
print("=" * 60)

v2_results = []
for i, query in enumerate(test_queries, 1):
    print(f"\n{"─"*60}")
    print(f"Query {i}: {query}\n")
    answer, tools_used, trace = run_agent(query, tools=my_tools, system_prompt=SYSTEM_PROMPT_V2)
    v2_results.append({"query": query, "answer": answer, "tools_used": tools_used, "trace": trace})
    print(f"🔧 Tools used: {tools_used}")
    if trace:
        print(f"\n📋 Agent Trace:")
        for t in trace:
            print(f"   [{t['call']}] {t['tool']}({t['args']}) → {str(t['result'])[:200]}")
    print(f"\n📝 Answer: {answer[:300]}")

In [ ]:
# ── Compare V1 vs V2 ────────────────────────────────────
v2_scores = evaluate_agent_results(v2_results, golden_set)
v2_pass_rate = sum(1 for s in v2_scores if s["passed"]) / len(v2_scores) * 100

print("=" * 60)
print("COMPARISON: V1 vs V2")
print("=" * 60)
print(f"V1 Pass Rate: {v1_pass_rate:.0f}%")
print(f"V2 Pass Rate: {v2_pass_rate:.0f}%")
print(f"Improvement:  {v2_pass_rate - v1_pass_rate:+.0f}%")

print("\nV2 Detailed Scores:")
print(pd.DataFrame(v2_scores).to_string(index=False))


---
## Step 8: Export Results


In [ ]:
# ── Export Golden Set ────────────────────────────────────
with open("day4_lab2_golden_set.json", "w") as f:
    json.dump(golden_set, f, indent=2)
print(f"✅ Exported golden set ({len(golden_set)} items)")


In [ ]:
# ── Export Results ───────────────────────────────────────
export_data = {
    "track": SELECTED_TRACK,
    "v1_pass_rate": v1_pass_rate,
    "v2_pass_rate": v2_pass_rate,
    "v1_results": [{"query": r["query"], "answer": r["answer"][:500]} for r in v1_results],
    "v2_results": [{"query": r["query"], "answer": r["answer"][:500]} for r in v2_results],
}
with open("day4_lab2_results.json", "w") as f:
    json.dump(export_data, f, indent=2)
print("✅ Exported results to day4_lab2_results.json")


In [ ]:
# ── Export Prompt Log ────────────────────────────────────
if PROMPT_LOG:
    import pandas as pd
    log_df = pd.DataFrame(PROMPT_LOG)
    log_df.to_csv("day4_lab2_prompt_log.csv", index=False)
    print(f"✅ Exported {len(log_df)} log entries to day4_lab2_prompt_log.csv")


---
## ✅ Checklist Before Assignment

- [ ] Selected a track and defined 3 tools
- [ ] Wrote system prompt with role, rules, and tool descriptions
- [ ] Tested agent on 5+ queries
- [ ] Created golden test set with 8+ scenarios
- [ ] Evaluated V1 and wrote error analysis
- [ ] Improved to V2 with better prompt/tools
- [ ] Compared V1 vs V2 metrics
- [ ] Exported all files (JSON, CSV)

**Next:** The Day 4 Assignment builds on this lab. Use the same track, tools, and data — your goal is to refine, evaluate, and document your agent to production-ready quality.
